# Using the PhenoMe pipeline with STED-FM models and data
## F-Actin conformations
---

## 1. Setup

In [10]:
import torch
from phenome import PhenoMe, get_metadata_from_path, load_dinov2_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pheno = PhenoMe(device=device, seed=42)

### Wrap the STED-FM model

In [14]:
from stedfm import get_pretrained_model_v2
from phenome.utils import ModelWrapper

class STEDFMWrapper(ModelWrapper):
    def __init__(self, device):
        # Load the pretrained STED-FM model
        model, cfg = get_pretrained_model_v2(
            name = "mae-lightning-small",
            weights = "MAE_SMALL_STED", 
            in_channels=1, # For single channel images
            as_classifier=True,
            # global_pool = "patch" # Returns the embeddings for all patches
        )

        # Initialize the parent class which moves the model to the device and sets it to eval()
        super().__init__(model=model, device=device)

    def _get_embeddings(self, tensor: torch.Tensor) -> torch.Tensor:
        out = self.model.forward_features(tensor)
        return out

wrapper = STEDFMWrapper(device=device)

in_channels 1
--- mae-lightning-small | (https://s3.valeria.science/flclab-foundation-models/models/mae-small-sted.zip) Downloading from URL and extracting zip ---

Loading state_dict from /Users/renaud/.stedfm/baselines/mae-small_STED/pl_checkpoint-999.pth
--- Loaded model mae-lightning-small with weights MAE_SMALL_STED ---
--- Added linear probe to all frozen blocks ---


### Using DINOv2

In [ ]:
wrapper = load_dinov2_model(model_name="dinov2_vits14_reg", device=device)

### Custom transforms for F-Actin dataset

In [15]:
import torchvision.transforms as T

# Transforms for STED-FM model
stedfm_transforms = T.Compose([
    T.ToPILImage(),
    T.Resize((224, 224), interpolation=0),
    T.Grayscale(num_output_channels=1),
    T.ToTensor(),
])

### Create 224x224x crops from the F-Actin dataset

In [ ]:
from data_handling import FActinConformationsDataset

# Create the dataset object
dataset = FActinConformationsDataset(
    path='/Users/renaud/datasets/factin-conformations-dataset',
    image_size=224,
    normalize='image',
    return_metadata=True,
    min_annotated_ratio=0.01
)

Processing dataset..: 100%|██████████| 79/79 [00:00<00:00, 481.04it/s]


In [ ]:
# Iterate through dataset and save individual images, masks and metadata.csv
import os

import numpy as np
import pandas as pd
import tifffile
from tqdm import tqdm

tiles_dir = "/Users/renaud/datasets/factin-conformations-dataset/tiles_224"
tiles_images_dir = os.path.join(tiles_dir, "images")
tiles_masks_dir = os.path.join(tiles_dir, "masks")
tiles_metadata_path = os.path.join(tiles_dir, "metadata.csv")

os.makedirs(tiles_images_dir, exist_ok=True)
os.makedirs(tiles_masks_dir, exist_ok=True)

rows = []
for image, mask, meta in tqdm(dataset, desc="Saving tiles"):
    stem = (
        f"img_{meta.get('dataset-idx')}"
    )

    image = np.asarray(image, dtype=np.float32)
    mask = np.asarray(mask, dtype=np.uint8)*255
    if image.ndim == 3 and image.shape[0] == 1:
        image = image[0]  # (1, H, W) -> (H, W)

    tifffile.imwrite(os.path.join(tiles_images_dir, f"{stem}.tif"), image)
    tifffile.imwrite(os.path.join(tiles_masks_dir, f"{stem}.tif"), mask)
    row_dict = {k: meta.get(k) for k in meta.keys() if k != 'mask'}
    row_dict["filename"] = stem
    rows.append(row_dict)

# 'filename' is the extension-less stem shared by the image and its mask, which is what
# make_dataframe_metadata_fn matches on in pheno.find_files().
tiles_metadata_df = pd.DataFrame(rows)
tiles_metadata_df.to_csv(tiles_metadata_path, index=False)

print(f"Saved {len(tiles_metadata_df)} tiles to {tiles_images_dir} and {tiles_masks_dir}")
print(f"Metadata written to {tiles_metadata_path}")
tiles_metadata_df.head()

Saving tiles: 100%|██████████| 114/114 [00:00<00:00, 1165.82it/s]

Saved 114 tiles to /Users/renaud/datasets/factin-conformations-dataset/tiles_ratio10_224/images and /Users/renaud/datasets/factin-conformations-dataset/tiles_ratio10_224/masks
Metadata written to /Users/renaud/datasets/factin-conformations-dataset/tiles_ratio10_224/metadata.csv


,image_name,image-idx,row,col,dataset-idx,mask_fraction_nanoclusters,mask_fraction_compact_assemblies,mask_fraction_protrusions,filename
0,0aab3e19-9942-3936-be0f-d524985dc3f9,1,0,224,0,0.008689,0.049845,0.105449,img_0
1,0aab3e19-9942-3936-be0f-d524985dc3f9,1,224,224,1,0.008929,0.008211,0.128009,img_1
2,0aab3e19-9942-3936-be0f-d524985dc3f9,1,224,288,2,0.006118,0.008211,0.090163,img_2
3,0aab3e19-9942-3936-be0f-d524985dc3f9,1,288,224,3,0.002671,0.008211,0.159220,img_3
4,0aab3e19-9942-3936-be0f-d524985dc3f9,1,288,288,4,0.000279,0.008211,0.119739,img_4


### Create dataset (find files)

In [16]:
# csv-based metadata
from phenome import make_dataframe_metadata_fn
from data_handling import minmax_preprocessing_fn
import pandas as pd

images_dir = "/Users/renaud/datasets/factin-conformations-dataset/tiles_224/images"
masks_dir = "/Users/renaud/datasets/factin-conformations-dataset/tiles_224/masks"
metadata_dir = "/Users/renaud/datasets/factin-conformations-dataset/tiles_224/metadata.csv"

metadata_df = pd.read_csv(metadata_dir)
metadata_fn = make_dataframe_metadata_fn(metadata_df)
file_df = pheno.find_files(images_dir, mask_dir=None, metadata_fn=metadata_fn)

print(f"Found {len(file_df)} images")



Found 288 files total.
DataFrame: 288 files, 12 columns.


Found 288 images


### Process images

In [17]:
# Process with STED-FM model

pheno.process_images(wrapper, 
                    force_rgb=False,
                    checkpoint_path='checkpoints/factin_dataset/factin_stedfm.h5',
                    preprocessing_fn=None,
                    custom_transformations=stedfm_transforms,
                    )
pheno.compute_properties(property_preset="complete", checkpoint_path="checkpoints/factin_dataset/factin_stedfm.h5")

Found 288 committed images; skipping 288 (already in checkpoint), 0 to process.
All images already processed. Loading from checkpoint.
Pipeline ready: 288 images available (lazy loading from checkpoints/factin_dataset/factin_stedfm.h5).
Properties: 288/288 matched in checkpoint (6 columns).


,image_index,image_path,image_name,intensity_entropy,intensity_max,intensity_mean,intensity_min,intensity_std,sharpness_metric
0,0,/Users/renaud/datasets/factin-conformations-da...,img_0.tif,4.262452,0.838323,0.041706,0.0,0.056183,0.280081
1,1,/Users/renaud/datasets/factin-conformations-da...,img_1.tif,3.138181,0.760479,0.026782,0.0,0.053234,0.284244
2,2,/Users/renaud/datasets/factin-conformations-da...,img_10.tif,2.375309,1.000000,0.024504,0.0,0.064071,0.265345
3,3,/Users/renaud/datasets/factin-conformations-da...,img_100.tif,2.314565,0.377273,0.009240,0.0,0.021616,0.239068
4,4,/Users/renaud/datasets/factin-conformations-da...,img_101.tif,1.650534,0.263636,0.004745,0.0,0.014196,0.232842
...,...,...,...,...,...,...,...,...,...
283,283,/Users/renaud/datasets/factin-conformations-da...,img_95.tif,3.918639,0.780952,0.028741,0.0,0.048540,0.299806
284,284,/Users/renaud/datasets/factin-conformations-da...,img_96.tif,4.138019,1.000000,0.038244,0.0,0.066460,0.312266
285,285,/Users/renaud/datasets/factin-conformations-da...,img_97.tif,4.555275,0.500000,0.038102,0.0,0.048062,0.272796
286,286,/Users/renaud/datasets/factin-conformations-da...,img_98.tif,4.034701,0.500000,0.030127,0.0,0.044903,0.280358


In [ ]:
# Process with DINO-v2 model

pheno.process_images(wrapper, 
                    force_rgb=True,
                    channel_mode='combined',
                    checkpoint_path='dinov2_80_1ch_checkpoint.h5',
                    preprocessing_fn=minmax_preprocessing_fn,
                    custom_transformations=dino_transforms,
                    )
pheno.compute_properties(property_preset="basic")

## 4. Cluster in embedding space

Clusters are **data-driven** groups in embedding space; they may or may not match your experimental labels. Set `n_clusters` between 2 and the number of images (see [Pipeline API](../../docs/src/content/docs/advanced/api/pipeline.md)). After clustering, labels are stored under metadata key `cluster`.

In [9]:
pheno.compute_clustering(n_clusters=3, clustering_method="kmeans")
print(pheno.get_available_property_keys())
print(pheno.get_available_metadata_keys())

pheno.plot_tsne(color_by="cluster")

Clustering: PCA extracted 100 components from 384 dimensions before k-means.
Clustering (kmeans): 3 clusters on 288 images (source=embeddings).


['intensity_entropy', 'intensity_max', 'intensity_mean', 'intensity_min', 'intensity_std', 'sharpness_metric']
['cluster', 'col', 'dataset-idx', 'filename', 'id', 'image-idx', 'image_name', 'mask_fraction_compact_assemblies', 'mask_fraction_nanoclusters', 'mask_fraction_protrusions', 'mask_path', 'row']


## Interactive Exploration

The pipeline provides an interactive dashboard using Plotly and ipywidgets. This allows you to explore the embedding space, highligh specific groups, and inspect individual images by clicking on dots.

Note: This requires a Jupyter environment with `ipywidgets` and `plotly` installed.


In [18]:
# Launch interactive explorer (if in Jupyter)
explorer = pheno.create_interactive_explorer()

## See Also

- [03_core_phenotyping_workflow.ipynb](03_core_phenotyping_workflow.ipynb): Core API tutorial
- [05_extending_phenome_plugins.ipynb](05_extending_phenome_plugins.ipynb): Writing custom plugins and model wrappers